# Alzheimer's Disease Classification from MRI Images

A transfer learning model (EfficientNetV2B0) with class weights that classifies brain MRI images into four classes:
`MildDemented`, `ModerateDemented`, `NonDemented` and `VeryMildDemented`.

Data and inspiration: [Classification of Alzheimers 95% TOTW (Kaggle)](https://www.kaggle.com/code/schorsi/classification-of-alzheimers-95-totw).

**How to run:** open the notebook in Google Colab with a GPU runtime, put the dataset in your Google Drive
(folder `Alzheimer_s Dataset` with the `train` and `test` subfolders) and set `DATA_DIR` below.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import hashlib
import os
import shutil
from datetime import datetime

import cv2
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import keras
import tensorflow as tf
from keras.applications import EfficientNetV2B0
from keras.optimizers import Adam
from keras.models import load_model, Sequential
from keras.utils import image_dataset_from_directory
from keras.layers import (Dense, Dropout, BatchNormalization, GlobalAveragePooling2D,
                          RandomRotation, RandomTranslation, RandomZoom, RandomFlip)
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (confusion_matrix, classification_report, accuracy_score,
                             precision_score, recall_score, f1_score)

In [ ]:
# Paths
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/data/input/Alzheimer_s Dataset'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR = os.path.join(DATA_DIR, 'test')
MODEL_DIR = '/content/drive/MyDrive/Colab Notebooks/data/output/model'
FIGURES_DIR = '/content/drive/MyDrive/Colab Notebooks/data/output/figures'  # charts for the README
os.makedirs(FIGURES_DIR, exist_ok=True)

# Constants
image_height = 208
image_width = 176
batch_size = 32
epochs_head = 15  # phase 1: only the new top layers are trained
epochs_fine_tuning = 25  # phase 2: the last layers of EfficientNet are also trained
unfreeze_layers = 30  # how many of the last EfficientNet layers are trained in phase 2
num_classes = 4
validation_split = 0.2  # part of the train folder used for validation
seed = 42  # same seed for training and validation, so the two parts do not overlap

## Data

- **Training:** 80% of the `train` folder. Image augmentation is part of the model and is applied only during training.
- **Validation:** the other 20% of the `train` folder. Used to choose the best epoch.
- **Test:** the separate `test` folder, never used during training. Used only for the final evaluation. Images identical to a training image are excluded first.

In [ ]:
# Exclude from the test set any image identical to a training image.
# The files in Drive are not changed: a clean copy of the test set is made on Colab.
def file_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

train_hashes = {file_hash(os.path.join(folder, name))
                for folder, _, names in os.walk(TRAIN_DIR) for name in names}

CLEAN_TEST_DIR = '/content/test_without_duplicates'
shutil.rmtree(CLEAN_TEST_DIR, ignore_errors=True)

removed = {}
for class_name in sorted(os.listdir(TEST_DIR)):
    source = os.path.join(TEST_DIR, class_name)
    if not os.path.isdir(source):
        continue
    target = os.path.join(CLEAN_TEST_DIR, class_name)
    os.makedirs(target)
    for name in os.listdir(source):
        path = os.path.join(source, name)
        if file_hash(path) in train_hashes:
            removed[class_name] = removed.get(class_name, 0) + 1
        else:
            shutil.copy(path, target)

print('Test images excluded because identical to a training image:', removed or 'none')

In [ ]:
train_ds = image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=validation_split,
    subset='training',
    seed=seed,
    image_size=(image_height, image_width),
    batch_size=batch_size,
    label_mode='categorical'
)

validation_ds = image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=validation_split,
    subset='validation',
    seed=seed,
    image_size=(image_height, image_width),
    batch_size=batch_size,
    label_mode='categorical'
)

# shuffle=False keeps predictions in the same order as the true labels
test_ds = image_dataset_from_directory(
    CLEAN_TEST_DIR,
    image_size=(image_height, image_width),
    batch_size=batch_size,
    label_mode='categorical',
    shuffle=False
)

# Class names in the same order as the label indices
class_names = train_ds.class_names
print(class_names)

In [ ]:
# Keep the images in memory after the first epoch, so they are not read again from Drive
train_ds = train_ds.cache().shuffle(1000, seed=seed).prefetch(tf.data.AUTOTUNE)
validation_ds = validation_ds.cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.cache().prefetch(tf.data.AUTOTUNE)

# Class weights: errors on rare classes count more during training
train_labels = np.concatenate([np.argmax(labels, axis=1) for _, labels in train_ds])
weights = compute_class_weight(class_weight='balanced', classes=np.arange(num_classes), y=train_labels)
class_weights = dict(enumerate(weights))

for index, weight in class_weights.items():
    print(f'{class_names[index]}: {weight:.2f}')

In [ ]:
# Visualizing a single image from each class
fig, axes = plt.subplots(1, num_classes, figsize=(15, 5))

for i, class_name in enumerate(class_names):
    class_directory = os.path.join(TEST_DIR, class_name)
    sample_image_path = os.path.join(class_directory, sorted(os.listdir(class_directory))[0])

    sample_image = plt.imread(sample_image_path)
    axes[i].imshow(sample_image, cmap='gray')
    axes[i].set_title(class_name)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## Model

In [ ]:
# Image augmentation (active only during training)
augmentation = Sequential([
    RandomRotation(20 / 360),
    RandomTranslation(0.2, 0.2),
    RandomZoom(0.2),
    RandomFlip('horizontal'),
], name='augmentation')

# Network pre-trained on ImageNet. It rescales the pixels by itself, so images are passed as 0-255
base_model = EfficientNetV2B0(include_top=False, weights='imagenet',
                              input_shape=(image_height, image_width, 3))
base_model.trainable = False

model = Sequential([
    keras.Input(shape=(image_height, image_width, 3)),
    augmentation,
    base_model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax'),
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Model checkpoint filename with timestamp
os.makedirs(MODEL_DIR, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_checkpoint_filename = os.path.join(MODEL_DIR, f'alzheimers_best_model_{timestamp}.keras')

# The same checkpoint is used in both phases, so the best model overall is kept
checkpoint = ModelCheckpoint(model_checkpoint_filename, monitor='val_loss', save_best_only=True, mode='min', verbose=1)

# Phase 1: EfficientNet is frozen, only the new top layers are trained
history_head = model.fit(
    train_ds,
    epochs=epochs_head,
    validation_data=validation_ds,
    class_weight=class_weights,
    callbacks=[checkpoint,
               EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
               ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-7, verbose=1)]
)

In [ ]:
# Phase 2: start from the best model and train also the last EfficientNet layers, slowly
model.load_weights(model_checkpoint_filename)

base_model.trainable = True
for layer in base_model.layers[:-unfreeze_layers]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, BatchNormalization):
        layer.trainable = False  # keep the statistics learned on ImageNet

model.compile(optimizer=Adam(learning_rate=0.00001), loss='categorical_crossentropy', metrics=['accuracy'])

history_fine = model.fit(
    train_ds,
    epochs=epochs_fine_tuning,
    validation_data=validation_ds,
    class_weight=class_weights,
    callbacks=[checkpoint,
               EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
               ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-7, verbose=1)]
)

In [ ]:
# Plot Training & Validation Accuracy and Loss for both phases
metric_history = {name: history_head.history[name] + history_fine.history[name]
                  for name in ('accuracy', 'val_accuracy', 'loss', 'val_loss')}
fine_tuning_start = len(history_head.history['loss']) - 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(metric_history['accuracy'], label='Training Accuracy')
axes[0].plot(metric_history['val_accuracy'], label='Validation Accuracy')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Training and Validation Accuracy')

axes[1].plot(metric_history['loss'], label='Training Loss')
axes[1].plot(metric_history['val_loss'], label='Validation Loss')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Loss')
axes[1].set_title('Training and Validation Loss')

for axis in axes:
    axis.axvline(fine_tuning_start, color='red', linestyle='--', label='Fine-tuning starts')
    axis.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'training_curves.png'), dpi=150)
plt.show()

## Evaluation on the test set

The test set has very different numbers of images per class, so the per-class results and the
macro averages say more than the overall accuracy.

In [ ]:
# Load the best saved model and make predictions on the test set
loaded_model = load_model(model_checkpoint_filename)

predictions = loaded_model.predict(test_ds)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.concatenate([np.argmax(labels, axis=1) for _, labels in test_ds])

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(true_classes, predicted_classes)

# Show the classes from the least to the most severe
clinical_order = ['NonDemented', 'VeryMildDemented', 'MildDemented', 'ModerateDemented']
order = [class_names.index(name) for name in clinical_order]
cm = cm[np.ix_(order, order)]

# Create a heatmap using seaborn
plt.figure(figsize=(8, 6))
sns.set(font_scale=1.2)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=clinical_order, yticklabels=clinical_order)
plt.title('Confusion Matrix', fontsize=16)
plt.xlabel('Predicted Labels', fontsize=12)
plt.ylabel('True Labels', fontsize=12)
plt.xticks(rotation=90, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

In [ ]:
# Calculate evaluation metrics
majority_share = np.bincount(true_classes).max() / len(true_classes)
print(f"Always predicting the most common class would give an accuracy of {majority_share:.3f}")
print(f"Accuracy: {accuracy_score(true_classes, predicted_classes):.3f}")
print(f"Precision (macro): {precision_score(true_classes, predicted_classes, average='macro', zero_division=0):.3f}")
print(f"Recall (macro): {recall_score(true_classes, predicted_classes, average='macro', zero_division=0):.3f}")
print(f"F1 Score (macro): {f1_score(true_classes, predicted_classes, average='macro', zero_division=0):.3f}")
print()
print(classification_report(true_classes, predicted_classes, target_names=class_names, digits=3, zero_division=0))

In [ ]:
# Predict one sample image per class from the test folder
num_images_to_display = 1
predictions_to_show = []
images = []

for class_name in class_names:
    class_directory = os.path.join(CLEAN_TEST_DIR, class_name)
    class_files = sorted(os.listdir(class_directory))
    for i in range(num_images_to_display):
        img = cv2.imread(os.path.join(class_directory, class_files[i]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (image_width, image_height)).astype('float32')

        # The model rescales the pixels itself, so the image is passed as 0-255
        prediction = loaded_model.predict(np.expand_dims(img, axis=0))[0]
        predictions_to_show.append((class_name, class_names[np.argmax(prediction)]))
        images.append(img / 255.0)

# Display the images with original and predicted classes in a single row
plt.figure(figsize=(15, 5))

for i, (img, (original_class_label, predicted_class_label)) in enumerate(zip(images, predictions_to_show)):
    plt.subplot(1, len(images), i + 1)
    plt.imshow(img)
    plt.title(f"Original: {original_class_label}\nPredicted: {predicted_class_label}")
    plt.axis('off')

plt.tight_layout()
plt.show()